# Final Three-Fish Strict Evaluation

Run this notebook to regenerate the strict input-local V2a evaluation outputs for the final manuscript scope: fish 1, fish 2, and fish 4. It intentionally excludes `220210_F1_run6`, which is outside the extension-plan final scope.

This notebook overwrites the strict tables under `outputs/evaluation/v2a-RSNs/<recording>/` for the three selected recordings, then writes aggregate tables under `outputs/evaluation/aggregate/`.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile


def find_project_root(start: Path | None = None) -> Path:
    """Find the ica-denoising project root from common Jupyter launch dirs."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    candidates.extend([start / "ica-denoising", *[parent / "ica-denoising" for parent in start.parents]])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "ica_denoising").exists():
            return candidate
    raise RuntimeError(f"Could not find ica-denoising project root from {start}")


PROJECT_ROOT = find_project_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

mpl_cache = Path(tempfile.gettempdir()) / "ica-denoising-matplotlib-cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

CONFIG_PATH = PROJECT_ROOT / "configs" / "evaluation.example.json"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")
print(f"Config:       {CONFIG_PATH}")
print(f"Output root:  {OUTPUT_ROOT}")
assert CONFIG_PATH.exists(), CONFIG_PATH

In [ ]:
DATASETS = [
    {
        "key": "v2a-RSNs/220119_F2_run11_fluorescence",
        "recording": "220119_F2_run11",
        "label": "fish 1",
    },
    {
        "key": "v2a-RSNs/220127_F4_run2_fluorescence",
        "recording": "220127_F4_run2",
        "label": "fish 2",
    },
    {
        "key": "v2a-RSNs/220210_F2_run5_fluorescence",
        "recording": "220210_F2_run5",
        "label": "fish 4",
    },
]

EXPECTED_PER_RECORDING = [
    "strict_trace_preservation_metrics.csv",
    "strict_behavior_fold_metrics.csv",
    "strict_behavior_predictions.csv",
    "strict_behavior_block_uncertainty.csv",
    "strict_trace_block_contributions.csv",
    "strict_causal_fold_metrics.csv",
    "strict_causal_oof_embeddings.csv",
    "strict_causal_block_contributions.csv",
    "strict_causal_sufficiency.csv",
    "strict_causal_sufficiency_block_contributions.csv",
    "nested_selection.csv",
    "variant_metadata.csv",
    "leakage_audit.csv",
    "run_manifest.json",
]

EXPECTED_AGGREGATE = [
    "recording_trace_preservation_metrics.csv",
    "recording_behavior_fold_metrics.csv",
    "recording_behavior_block_uncertainty.csv",
    "recording_trace_block_contributions.csv",
    "recording_causal_fold_metrics.csv",
    "recording_causal_block_contributions.csv",
    "recording_causal_sufficiency.csv",
    "recording_causal_sufficiency_block_contributions.csv",
    "recording_nested_selection.csv",
    "recording_primary_effects.csv",
    "fish_primary_effects.csv",
]

for item in DATASETS:
    print(f"{item['label']}: {item['key']} -> outputs/evaluation/v2a-RSNs/{item['recording']}")

## Run Evaluation

This is the long-running cell. It runs the strict evaluation once per final-scope recording and then writes the cross-recording aggregate tables.

In [ ]:
from ica_denoising.evaluation_runner import run_dataset_evaluation, write_recording_aggregate

completed = {}
for item in DATASETS:
    key = item["key"]
    print(f"\n=== Running {item['label']}: {key} ===", flush=True)
    paths = run_dataset_evaluation(
        key,
        project_root=PROJECT_ROOT,
        config_path=CONFIG_PATH,
        output_root=OUTPUT_ROOT,
    )
    completed[key] = paths
    print(f"Finished {key}", flush=True)
    print(f"Manifest: {paths['run_manifest']}", flush=True)

aggregate_paths = write_recording_aggregate(
    completed,
    project_root=PROJECT_ROOT,
    output_root=OUTPUT_ROOT,
)

print("\nAggregate outputs:")
for label, path in aggregate_paths.items():
    print(f"{label}: {path}")

## Verify Expected Outputs

Run this after the evaluation cell. It fails fast if any per-recording or aggregate output expected by the manuscript runbook is missing.

In [ ]:
missing = []

for item in DATASETS:
    recording_dir = OUTPUT_ROOT / "v2a-RSNs" / item["recording"]
    for filename in EXPECTED_PER_RECORDING:
        path = recording_dir / filename
        if not path.exists():
            missing.append(path)

aggregate_dir = OUTPUT_ROOT / "aggregate"
for filename in EXPECTED_AGGREGATE:
    path = aggregate_dir / filename
    if not path.exists():
        missing.append(path)

if missing:
    print("Missing outputs:")
    for path in missing:
        print(f"- {path}")
    raise FileNotFoundError(f"Missing {len(missing)} expected output(s)")

print("All expected final three-fish strict evaluation outputs are present.")